In [ ]:
import json
import pandas as pd
from collections import defaultdict

In [ ]:
# --- Define the High-Level Harm Categories and their associated granular risks ---
harm_categories_guide = {
    "Malicious Use & Security": ["Misuse", "Harmful Application", "Adversarial Data", "Security Vulnerability", "Lack of Adversarial Robustness", "Gaming Vulnerability", "System Manipulation", "Prompt Injection", "Lack of Authenticity Assurance", "Unauthorized Data", "Malicious Marketing"],
    "Fairness, Bias & Discrimination": ["Algorithmic Bias", "Distributional Bias", "Dataset Imbalance", "Problematic Features", "Inappropriate Training Content", "Discrimination and Toxicity"],
    "Safety, Robustness & Reliability": ["Generalization Failure", "Robustness Failure", "Hardware Failure", "Software Bug", "Overfitting", "Underfitting", "Concept Drift", "Covariate Shift", "Black Swan Event", "Scaling Limitations", "Latency Issues", "Untested Deployment"],
    "Privacy & Data Protection": ["Privacy Concerns", "Inadequate Anonymization", "Unsafe Exposure or Access"],
    "Transparency & Explainability": ["Lack of Transparency", "Lack of Explainability"],
    "Societal & Economic Impact": ["Misinformation Generation Hazard", "Overpersonalization", "Exposure to toxic content"],
    "Human-Computer Interaction & Autonomy": ["Lack of Safety Protocols", "Human Error", "Lack of Capability Control", "Faulty Interface or Instructions", "Lack of Corrigibility", "Lack of Interruptability"],
    "Data Quality & Integrity": ["Limited Dataset", "Data or Labelling Noise", "Inadequate Verification", "Incomplete Data Attribute Capture", "Inadequate Data Augmentation", "Faulty or Inadequate Preprocessing", "Inadequate Provenance", "Inadequate Data Sampling", "Outdated Ground Truth"],
    "System & Task Mismatch": ["Context Misidentification", "Task Mismatch", "Misaligned Objective", "Underspecification", "Misconfigured Threshold", "Tuning Issues", "Domain Adaptation Deficit"]
}

In [ ]:
# --- Create a "reverse" map for fast lookups ---
# This dictionary will map a granular risk (e.g., "Tuning Issues") to its high-level category.
risk_to_category_map = {}
for category, risks in harm_categories_guide.items():
    for risk in risks:
        # Add a mapping for both the original risk title and a lowercase version for broader matching
        risk_to_category_map[risk] = category
        risk_to_category_map[risk.lower()] = category


print("Harm category map created successfully.")
print(f"Mapped {len(risk_to_category_map)} granular risk terms.")

Harm category map created successfully.
Mapped 118 granular risk terms.


In [35]:
# --- Load and process the incident data ---
print("\nLoading unified_incidents.json...")
with open('unified_incidents.json', 'r') as f:
    incidents_data = json.load(f)
print(f"Successfully loaded {len(incidents_data)} incidents.")

# Create a DataFrame from the raw data
df = pd.DataFrame(incidents_data)

# Take the first 200 for our gold dataset
df_manual = df.head(200).copy()
print(f"Selected the first {len(df_manual)} incidents for the gold dataset.")


Loading unified_incidents.json...
Successfully loaded 1238 incidents.
Selected the first 200 incidents for the gold dataset.


In [36]:
#2. Run Automated Classification Logic

def classify_incident_programmatically(incident):
    """
    Classifies an incident by searching for known risk keywords in its classification data.
    It follows a prioritized search order.
    """
    classifications = incident.get('classifications', {})
    if not classifications:
        return "Uncategorized - No classification data"

    # --- Prioritized search order ---
    # 1. GMF Known AI Technical Failure (very specific)
    # 2. GMF Potential AI Technical Failure
    # 3. MIT Risk Subdomain (often descriptive)
    # 4. MIT Risk Domain

    # Check GMF Known AI Technical Failure
    gmf_known_failures = classifications.get('classifications_GMF', {}).get('Known AI Technical Failure', '')
    if gmf_known_failures:
        # The field can be a comma-separated string
        for failure in gmf_known_failures.split(', '):
            if failure in risk_to_category_map:
                return risk_to_category_map[failure]

    # Check GMF Potential AI Technical Failure
    gmf_potential_failures = classifications.get('classifications_GMF', {}).get('Potential AI Technical Failure', '')
    if gmf_potential_failures:
        for failure in gmf_potential_failures.split(', '):
            if failure in risk_to_category_map:
                return risk_to_category_map[failure]

    # Check MIT Risk Subdomain
    mit_subdomain = classifications.get('classifications_MIT', {}).get('Risk Subdomain', '')
    if mit_subdomain:
        # Clean the string, e.g., "1.2. Exposure to toxic content" -> "Exposure to toxic content"
        cleaned_subdomain = mit_subdomain.split('. ')[-1]
        if cleaned_subdomain.lower() in risk_to_category_map:
            return risk_to_category_map[cleaned_subdomain.lower()]

    # Check MIT Risk Domain
    mit_domain = classifications.get('classifications_MIT', {}).get('Risk Domain', '')
    if mit_domain:
        cleaned_domain = mit_domain.split('. ')[-1]
        if cleaned_domain.lower() in risk_to_category_map:
            return risk_to_category_map[cleaned_domain.lower()]

    return "Uncategorized"


# --- Apply the classification function to the DataFrame ---
print("Applying automated classification to the dataset...")
df_manual['harm_category'] = df_manual.apply(classify_incident_programmatically, axis=1)
print("Classification complete.")

# --- Display a summary of the results ---
print("\n--- Classification Summary ---")
print(df_manual['harm_category'].value_counts())

print("\n--- Examples of Classified Incidents ---")
# Display relevant columns to verify the logic
display_cols = ['incident_id', 'title', 'harm_category']
print(df_manual[display_cols].head(20))


Applying automated classification to the dataset...
Classification complete.

--- Classification Summary ---
harm_category
Safety, Robustness & Reliability         51
Fairness, Bias & Discrimination          40
Malicious Use & Security                 26
System & Task Mismatch                   22
Data Quality & Integrity                 19
Transparency & Explainability            14
Privacy & Data Protection                 9
Human-Computer Interaction & Autonomy     7
Societal & Economic Impact                7
Uncategorized                             5
Name: count, dtype: int64

--- Examples of Classified Incidents ---
   incident_id                                              title  \
0            1  Google’s YouTube Kids App Presents Inappropria...   
1            2  Warehouse robot ruptures can of bear spray and...   
2            3  Crashes with Maneuvering Characteristics Augme...   
3            4               Uber AV Killed Pedestrian in Arizona   
4            5         C

In [37]:
# Save and Download the "Gold Dataset"
from google.colab import files

# Extract the essential columns for the final CSV
final_columns = [
    'incident_id',
    'date',
    'title',
    'description', # The general, shorter description
    'harm_category'
]

# Create the final DataFrame
df_gold = df_manual[final_columns].copy()

# --- Save the DataFrame to a CSV file ---
output_filename = 'manual_classified_incidents.csv'
df_gold.to_csv(output_filename, index=False)

print(f"Successfully saved the gold dataset as '{output_filename}'")
print("\nThis file contains your 200 programmatically classified incidents and is ready for the next stage.")

# --- Download the file to your local machine ---
print(f"Downloading the file to your computer...")
files.download(output_filename)

Successfully saved the gold dataset as 'manual_classified_incidents.csv'

This file contains your 200 programmatically classified incidents and is ready for the next stage.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>